# Fine-Tuning ALBERT on GoEmotions (Multi-Label, Train Split Only)

This notebook fine-tunes `albert-base-v2` on the GoEmotions `simplified` configuration, which keeps multi-label examples. Training uses only the **train** split for gradient updates. The **validation** split is used for early stopping, checkpoint selection, and decision-threshold tuning. The **test** split is evaluated once at the end.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

### Cell 1: Check the GPU and mount Google Drive

In [17]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

from google.colab import drive
drive.mount("/content/drive")

name, memory.total [MiB]
Tesla T4, 15360 MiB

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Cell 2: Install dependencies

Colab already includes PyTorch and `transformers`, so only the extras are installed. If Colab reports a dependency conflict and asks you to restart, restart the runtime and run from Cell 3 onward.

In [18]:
%pip install -q "transformers>=4.46" "datasets>=2.19" "accelerate>=0.30" sentencepiece

### Cell 3: Imports, reproducibility, and environment settings

In [19]:
import os
import json
import random
import warnings
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    hamming_loss,
    classification_report,
)

warnings.filterwarnings("ignore", category=UserWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(torch.cuda.get_device_name(0))

Using device: cuda
Tesla T4


### Cell 4: Load GoEmotions (simplified configuration)

In [20]:
MODEL_NAME = "albert-base-v2"
DATASET_NAME = "google-research-datasets/go_emotions"

raw = load_dataset(DATASET_NAME, "simplified")
print(raw)

label_names = raw["train"].features["labels"].feature.names
NUM_LABELS = len(label_names)
print(f"Number of classes: {NUM_LABELS}")

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})
Number of classes: 28


### Cell 5: Convert labels to multi-hot vectors

Each example becomes a 28-dimensional float vector with a 1 for every emotion it carries. Single-label and multi-label examples are handled the same way.

In [21]:
def to_multi_hot(example):
    vec = [0.0] * NUM_LABELS
    for idx in example["labels"]:
        vec[idx] = 1.0
    example["labels"] = vec
    return example

ds = raw.map(to_multi_hot, remove_columns=["id"], desc="Multi-hot encoding")
del raw

# Report multi-label statistics per split
for split in ds:
    arr = np.array(ds[split]["labels"])
    n_multi = int((arr.sum(axis=1) > 1).sum())
    print(f"{split:<11} size={len(arr):>6}  multi-label={n_multi:>5} ({100*n_multi/len(arr):.1f}%)")

# Check for text overlap between train and the held-out splits
test_texts = set(ds["test"]["text"])
val_texts = set(ds["validation"]["text"])

overlap_test = sum(t in test_texts for t in ds["train"]["text"])
overlap_val = sum(t in val_texts for t in ds["train"]["text"])
print(f"\nTrain rows whose text also appears in test:       {overlap_test}")
print(f"Train rows whose text also appears in validation: {overlap_val}")

# Remove those rows from TRAIN only. Validation and test are left untouched.
n_before = len(ds["train"])
ds["train"] = ds["train"].filter(
    lambda ex: ex["text"] not in test_texts and ex["text"] not in val_texts,
    desc="Removing train/eval text overlap",
)
print(f"Train size: {n_before} -> {len(ds['train'])}")

# Confirm no overlap remains
assert set(ds["train"]["text"]).isdisjoint(test_texts), "Train/test overlap still present"
assert set(ds["train"]["text"]).isdisjoint(val_texts), "Train/validation overlap still present"
del test_texts, val_texts

# Per-label counts in the cleaned training split
train_label_counts = np.array(ds["train"]["labels"]).sum(axis=0).astype(int)
for i, name in enumerate(label_names):
    print(f"{i:2d} {name:<15} {train_label_counts[i]}")

train       size= 43410  multi-label= 7102 (16.4%)
validation  size=  5426  multi-label=  878 (16.2%)
test        size=  5427  multi-label=  837 (15.4%)

Train rows whose text also appears in test:       79
Train rows whose text also appears in validation: 84
Train size: 43410 -> 43284
 0 admiration      4122
 1 amusement       2326
 2 anger           1565
 3 annoyance       2467
 4 approval        2934
 5 caring          1085
 6 confusion       1363
 7 curiosity       2186
 8 desire          641
 9 disappointment  1267
10 disapproval     2022
11 disgust         791
12 embarrassment   303
13 excitement      848
14 fear            594
15 gratitude       2620
16 grief           77
17 joy             1445
18 love            2070
19 nervousness     164
20 optimism        1577
21 pride           111
22 realization     1109
23 relief          153
24 remorse         544
25 sadness         1322
26 surprise        1059
27 neutral         14197


### Cell 6: Tokenization

In [22]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LENGTH = 128

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized = ds.map(tokenize_fn, batched=True, remove_columns=["text"], desc="Tokenizing")
tokenized.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Cell 7: Build the model for multi-label classification

`problem_type="multi_label_classification"` makes the Trainer use `BCEWithLogitsLoss`, which treats each label as an independent yes/no decision.

In [23]:
id2label = {i: n for i, n in enumerate(label_names)}
label2id = {n: i for i, n in enumerate(label_names)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id,
)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: albert-base-v2
Key                          | Status     | 
-----------------------------+------------+-
predictions.LayerNorm.weight | UNEXPECTED | 
predictions.bias             | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
predictions.decoder.bias     | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 11,705,116


### Cell 8: Multi-label metrics

Logits are thresholded at 0, which is equivalent to a sigmoid probability of 0.5. Macro F1 is the checkpoint-selection metric because it weights rare emotions as heavily as common ones.

In [24]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = np.asarray(logits)
    labels = np.asarray(labels).astype(int)
    preds = (logits > 0).astype(int)

    return {
        "subset_accuracy": accuracy_score(labels, preds),
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "samples_f1": f1_score(labels, preds, average="samples", zero_division=0),
        "hamming_loss": hamming_loss(labels, preds),
    }

### Cell 9: Training configuration (Colab-friendly)

Checkpoints go to Google Drive so they survive runtime disconnects. `group_by_length` batches similar-length texts together, which speeds up training with dynamic padding.

In [25]:
import math

DRIVE_ROOT = "/content/drive/MyDrive/albert-goemotions"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
FINAL_DIR = f"{DRIVE_ROOT}/final"
os.makedirs(CKPT_DIR, exist_ok=True)

NUM_EPOCHS = 5
TRAIN_BATCH = 32
EVAL_BATCH = 64
WARMUP_FRACTION = 0.1

# Compute warmup steps explicitly, since warmup_ratio is not accepted in newer versions
steps_per_epoch = math.ceil(len(tokenized["train"]) / TRAIN_BATCH)
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = int(WARMUP_FRACTION * total_steps)
print(f"Steps per epoch: {steps_per_epoch}, total: {total_steps}, warmup: {warmup_steps}")

training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    group_by_length=True,
    fp16=(device == "cuda"),
    dataloader_num_workers=2,
    logging_steps=100,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

Steps per epoch: 1353, total: 6765, warmup: 676


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'group_by_length'

### Cell 10: Fine-tune (resumes automatically if a previous run was interrupted)

In [ ]:
last_ckpt = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
if last_ckpt:
    print(f"Resuming from {last_ckpt}")

train_result = trainer.train(resume_from_checkpoint=last_ckpt)
print(train_result)

### Cell 11: Tune the decision threshold on the validation split

This step searches for a global threshold using validation data only. The test split is not involved.

In [ ]:
val_output = trainer.predict(tokenized["validation"], metric_key_prefix="val")
val_logits = np.asarray(val_output.predictions)
val_labels = np.asarray(val_output.label_ids).astype(int)
val_probs = 1 / (1 + np.exp(-val_logits))

best_thr, best_f1 = 0.5, -1.0
for thr in np.arange(0.10, 0.91, 0.02):
    f1 = f1_score(val_labels, (val_probs >= thr).astype(int), average="macro", zero_division=0)
    if f1 > best_f1:
        best_thr, best_f1 = float(thr), float(f1)

print(f"Best validation threshold: {best_thr:.2f} (macro F1 = {best_f1:.4f})")

### Cell 12: Final evaluation on the test split (run once)

In [ ]:
test_output = trainer.predict(tokenized["test"], metric_key_prefix="test")
test_logits = np.asarray(test_output.predictions)
test_labels = np.asarray(test_output.label_ids).astype(int)
test_probs = 1 / (1 + np.exp(-test_logits))
test_preds = (test_probs >= best_thr).astype(int)

print(f"Test subset accuracy: {accuracy_score(test_labels, test_preds):.4f}")
print(f"Test micro F1:        {f1_score(test_labels, test_preds, average='micro', zero_division=0):.4f}")
print(f"Test macro F1:        {f1_score(test_labels, test_preds, average='macro', zero_division=0):.4f}")
print(f"Test samples F1:      {f1_score(test_labels, test_preds, average='samples', zero_division=0):.4f}")
print(f"Test Hamming loss:    {hamming_loss(test_labels, test_preds):.4f}")
print()
print(classification_report(test_labels, test_preds, target_names=label_names,
                            digits=3, zero_division=0))

### Cell 13: Save the model, tokenizer, and threshold to Drive

In [ ]:
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
with open(f"{FINAL_DIR}/inference_config.json", "w") as f:
    json.dump({"threshold": best_thr, "max_length": MAX_LENGTH, "label_names": label_names}, f, indent=2)
print(f"Saved to {FINAL_DIR}")

### Cell 14: Multi-label inference example

In [ ]:
with open(f"{FINAL_DIR}/inference_config.json") as f:
    cfg = json.load(f)

inf_tokenizer = AutoTokenizer.from_pretrained(FINAL_DIR)
inf_model = AutoModelForSequenceClassification.from_pretrained(FINAL_DIR).to(device).eval()

def predict_emotions(texts, threshold=cfg["threshold"]):
    enc = inf_tokenizer(texts, truncation=True, max_length=cfg["max_length"],
                        padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = torch.sigmoid(inf_model(**enc).logits).cpu().numpy()

    results = []
    for p in probs:
        idx = np.where(p >= threshold)[0]
        if len(idx) == 0:
            idx = [int(p.argmax())]
        idx = sorted(idx, key=lambda i: -p[i])
        results.append([(cfg["label_names"][i], float(p[i])) for i in idx])
    return results

samples = [
    "I can't believe they cancelled the concert, I was so excited!",
    "Thanks so much for helping me move, I really appreciate it.",
    "I just found out my dog got lost and I don't know where to look.",
    "I'm thrilled but also terrified about starting the new job tomorrow.",
]
for text, emotions in zip(samples, predict_emotions(samples)):
    print(text)
    for name, score in emotions:
        print(f"   {name:<15} {score:.3f}")
    print()